# External Validation, Calibration, DCA, and Subgroup Analysis

*Generated 2025-07-25*

This notebook applies the locked model to an external (temporal) dataset and produces:
1. Confusion matrix & metrics
2. ROC & AUROC
3. Calibration plot (deciles)
4. Decision curve analysis (DCA)
5. Subgroup AUROC & F1 with bootstrap CIs

Edit file paths and subgroup definitions as needed.

## 0. Imports

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, f1_score)
from sklearn.calibration import calibration_curve
import seaborn as sns
import joblib, json, os
from collections import defaultdict

## 1. Load model & external dataset

In [ ]:
# === USER SETTINGS ===
EXT_CSV = 'data/external_data.csv'   # replace
TARGET_COL = 'NPB300'
ID_COL = None

# Artifacts
MODEL_PKL = 'models/model.pkl'
SCALER_PKL = 'models/standard_scaler.pkl'
FEAT_JSON  = 'models/feature_order.json'
THRESH = 0.50

# Load
df_ext = pd.read_csv(EXT_CSV)
y_ext = df_ext[TARGET_COL].astype(int)
if ID_COL and ID_COL in df_ext.columns:
    df_id = df_ext[ID_COL]
else:
    df_id = None

with open(FEAT_JSON,'r') as f:
    features = json.load(f)
X_ext = df_ext[features]

scaler = joblib.load(SCALER_PKL)
model  = joblib.load(MODEL_PKL)

X_ext_s = scaler.transform(X_ext)
proba_ext = model.predict_proba(X_ext_s)[:,1]
pred_ext  = (proba_ext >= THRESH).astype(int)


## 2. Confusion matrix & metrics

In [ ]:
cm = confusion_matrix(y_ext, pred_ext)
class_names = ['<=300','>300']
plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Observed'); plt.title('External Confusion Matrix')
plt.tight_layout(); plt.savefig('docs/confusion_external.png', dpi=300); plt.show()

print(classification_report(y_ext, pred_ext, target_names=class_names))
print('AUROC:', roc_auc_score(y_ext, proba_ext))

## 3. ROC curve

In [ ]:
fpr, tpr, thr = roc_curve(y_ext, proba_ext)
auc = roc_auc_score(y_ext, proba_ext)
plt.figure()
plt.plot(fpr, tpr, lw=2, label=f'External ROC (AUC={auc:.2f})')
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlim([0,1]); plt.ylim([0,1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.tight_layout(); plt.savefig('docs/roc_external.png', dpi=300); plt.show()

## 4. Calibration plot (deciles)

In [ ]:
# Create deciles
df_cal = pd.DataFrame({'prob': proba_ext, 'y': y_ext})
df_cal['bin'] = pd.qcut(df_cal['prob'], 10, duplicates='drop')
cal = df_cal.groupby('bin').agg(obs_rate=('y','mean'), pred_mean=('prob','mean'), n=('y','size'))

plt.figure()
plt.plot([0,1],[0,1],'--',color='gray')
plt.scatter(cal['pred_mean'], cal['obs_rate'], s=cal['n']*2, alpha=0.7)
for i,(x,y_) in enumerate(zip(cal['pred_mean'], cal['obs_rate'])):
    plt.text(x+0.01,y_,str(i+1), fontsize=8)
plt.xlabel('Mean predicted probability')
plt.ylabel('Observed event rate')
plt.title('Calibration (deciles) - External')
plt.tight_layout(); plt.savefig('docs/calibration_external.png', dpi=300); plt.show()
cal

## 5. Decision curve analysis (DCA)

In [ ]:
def net_benefit(y_true, y_prob, thresh):
    y_pred = (y_prob >= thresh).astype(int)
    tp = ((y_pred==1)&(y_true==1)).sum()
    fp = ((y_pred==1)&(y_true==0)).sum()
    n  = len(y_true)
    t = thresh/(1-thresh)
    return tp/n - fp/n * t

ths = np.linspace(0.05,0.60,56)
nb_model = [net_benefit(y_ext, proba_ext, t) for t in ths]
# treat-all: predict positive for everyone
y_all = np.ones_like(y_ext)
nb_all = [net_benefit(y_ext, np.ones_like(proba_ext), t) for t in ths]
# treat-none is 0

plt.figure()
plt.plot(ths, nb_model, label='Model')
plt.plot(ths, nb_all, '--', label='Treat-all')
plt.axhline(0, color='gray', linestyle='--', label='Treat-none')
plt.xlabel('Threshold probability')
plt.ylabel('Net benefit')
plt.title('Decision Curve Analysis (External)')
plt.legend()
plt.tight_layout(); plt.savefig('docs/dca_external.png', dpi=300); plt.show()


## 6. Subgroup analysis (AUROC & F1 with bootstrap CIs)

In [ ]:
from sklearn.utils import resample

def boot_ci(metric_fn, y, p, n=2000, seed=42):
    rng = np.random.default_rng(seed)
    vals=[]
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        vals.append(metric_fn(y[idx], p[idx]))
    return np.percentile(vals,[2.5,97.5])

# === Define subgroups (edit as needed) ===
subgroups = {
    'Age <65':  df_ext['Age'] < 65,
    'Age ≥65':  df_ext['Age'] >= 65,
    'Male':     df_ext['Sex'] == 1 if 'Sex' in df_ext.columns else pd.Series(False,index=df_ext.index),
    'Female':   df_ext['Sex'] == 0 if 'Sex' in df_ext.columns else pd.Series(False,index=df_ext.index),
    'eGFR <60': df_ext['eGFR'] < 60,
    'eGFR ≥60': df_ext['eGFR'] >= 60,
}

rows=[]
for name, m in subgroups.items():
    m = m.fillna(False)
    y_g = y_ext[m]
    p_g = proba_ext[m]
    if y_g.size < 30:
        continue
    auc = roc_auc_score(y_g, p_g)
    f1  = f1_score(y_g, (p_g>=THRESH).astype(int))
    lo_auc, hi_auc = boot_ci(lambda yt, pr: roc_auc_score(yt, pr), y_g.values, p_g.values)
    rows.append({'Group':name,'N':y_g.size,'AUROC':auc,'AUROC_lo':lo_auc,'AUROC_hi':hi_auc,'F1':f1})

sub_df = pd.DataFrame(rows)
sub_df

In [ ]:
# Plot
import matplotlib.pyplot as plt
x = np.arange(len(sub_df))
au = sub_df['AUROC'].values
err_lo = au - sub_df['AUROC_lo'].values
err_hi = sub_df['AUROC_hi'].values - au
f1v = sub_df['F1'].values

fig, ax = plt.subplots(figsize=(6,3.2))
ax.bar(x, au, width=0.6, color='#4C78A8', alpha=0.85, label='AUROC')
ax.errorbar(x, au, yerr=[err_lo, err_hi], fmt='none', ecolor='black', capsize=3)
ax.scatter(x, f1v, color='black', s=35, label='F1-score', zorder=3)
ax.set_ylim(0,1)
ax.set_xticks(x)
ax.set_xticklabels(sub_df['Group'], rotation=35, ha='right')
ax.set_ylabel('Performance')
ax.set_title('Subgroup discrimination')
ax.legend(frameon=False, ncol=2, loc='lower center', bbox_to_anchor=(0.5,-0.35))
ax.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout(); plt.savefig('docs/subgroup_external.png', dpi=300); plt.show()

---
**End of notebook**